# TirraMind — HetTGN GNN Retrain on Kaggle or CPU

This notebook resumes training from **epoch_003.pt** and completes epochs 4–5.
It uses **CUDA when available** and automatically falls back to **CPU** when no GPU is attached.

## Before Running — Two Things to Do

### Step 1 — Make sure two datasets are attached

**Dataset 1: `tirramind-code`**
It must contain these top-level directories somewhere under the mounted dataset tree:
- `agent/`
- `scripts/`

**Dataset 2: `tirramind-data`**
It must contain a directory with:
- `pipeline.db`
- `checkpoints/epoch_003.pt`

The notebook does **not** read zip files at runtime. It searches the mounted Kaggle dataset tree for those extracted directories/files directly.

### Step 2 — Attach datasets to THIS notebook

In the Kaggle notebook editor, right panel → **Data** → **Add Dataset**:
- Search `tirramind-code` → Add
- Search `tirramind-data` → Add

Optional but recommended: set **Accelerator → GPU T4 x1** in Settings.
If you do not attach a GPU, the notebook will run on CPU instead.

---
After training, download `gnn_model.pt` and `epoch_004.pt` / `epoch_005.pt` from the **Output** tab.


## 1. Install and Import Required Libraries

Install `torch-geometric` (and its sparse/scatter kernels) matching the Kaggle-provided PyTorch version.



In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

# torch-geometric — core dependency not bundled in Kaggle's default image
pip("torch-geometric==2.7.0")

# scatter/sparse kernels — pick wheels matching the active torch + device runtime
import torch

torch_ver = torch.__version__.split("+")[0]
cuda_runtime = torch.version.cuda
cuda_tag = f"cu{cuda_runtime.replace('.', '')}" if cuda_runtime else "cpu"
wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
print(f"Installing PyG extras for torch={torch_ver}, runtime={cuda_tag}")
pip("torch-scatter", "torch-sparse", "-f", wheel_url)

# other lightweight deps
pip("tqdm", "rich")

print("All dependencies installed.")


## 2. Setup Working Directory

**Code** is cloned directly from GitHub (always latest — no dataset re-upload needed for code changes).
To enable this, add a Kaggle Secret named `GITHUB_TOKEN` with a fine-grained personal access token
that has **read** access to the `tirramind_v1` repo.

Go to: Kaggle → Settings → Secrets → Add New Secret → Name: `GITHUB_TOKEN`

If no secret is set, falls back to the `tirramind-code` dataset (backward compatible).

**Data** (`pipeline.db` + epoch checkpoints) is still read from the `tirramind-data` dataset —
these are large binary files that change infrequently and are not in git.



In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. CODE: git clone (preferred) or fall back to dataset ───────────────────
GITHUB_REPO = "savabs/tirramind"  # GitHub username/repo

_cloned_from_git = False
try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    _repo_url = f"https://{_token}@github.com/{GITHUB_REPO}.git"

    # Fresh clone every run — always gets the latest trainer.py / scripts
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    subprocess.run(
        ["git", "clone", "--depth=1", _repo_url, str(WORK_DIR)],
        check=True,
        capture_output=True,
    )
    print(f"✓ Cloned {GITHUB_REPO} → {WORK_DIR}  (latest commit, no dataset re-upload needed)")
    _cloned_from_git = True

except Exception as _e:
    print(f"Git clone skipped ({_e.__class__.__name__}: {_e})")
    print("  Falling back to tirramind-code dataset — add GITHUB_TOKEN secret to avoid this.")

    def find_code_root(root: str = "/kaggle/input") -> Path | None:
        for dirpath, dirs, _ in os.walk(root):
            if {"agent", "scripts"}.issubset(set(dirs)):
                return Path(dirpath)
        return None

    code_root = find_code_root()
    assert code_root is not None, (
        "No GITHUB_TOKEN secret and no tirramind-code dataset found. "
        "Either add the secret or attach the dataset."
    )
    for name in ("agent", "scripts"):
        dest = WORK_DIR / name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(code_root / name, dest)
        print(f"  Copied {name}/ from dataset")

# ── 2. DATA: pipeline.db + checkpoints always come from the dataset ──────────
def find_data_root(root: str = "/kaggle/input") -> Path | None:
    for dirpath, dirs, files in os.walk(root):
        if "pipeline.db" in set(files) and "checkpoints" in set(dirs):
            return Path(dirpath)
    return None

data_root = find_data_root()
assert data_root is not None, (
    "Could not find tirramind-data dataset containing pipeline.db and checkpoints/. "
    "Attach the dataset in the right panel → Data → Add Dataset."
)

pipeline_db = data_root / "pipeline.db"
ckpt_src    = data_root / "checkpoints"
checkpoint_files = sorted(ckpt_src.glob("epoch_*.pt"))
assert checkpoint_files, f"No epoch_*.pt files found in {ckpt_src}"

pipeline_dir = WORK_DIR / ".tirra_pipeline"
ckpt_dir     = pipeline_dir / "checkpoints"
pipeline_dir.mkdir(exist_ok=True)
ckpt_dir.mkdir(exist_ok=True)

shutil.copy2(pipeline_db, pipeline_dir / "pipeline.db")
print(f"✓ pipeline.db  ({pipeline_db.stat().st_size // 1_000_000} MB)")

for ckpt in checkpoint_files:
    shutil.copy2(ckpt, ckpt_dir / ckpt.name)
    print(f"✓ {ckpt.name}  ({ckpt.stat().st_size // 1_000_000} MB)")

# ── 3. Patch pipeline __init__ to avoid eager APScheduler import ─────────────
pipeline_init = WORK_DIR / "agent" / "pipeline" / "__init__.py"
pipeline_init.write_text(
    '"""TirraMind — Pipeline Layer (Deterministic DAG Scheduler)."""\n\n'
    "from agent.pipeline.storage_backend import (\n"
    "    PostgresBackend,\n    SQLiteBackend,\n    StorageBackend,\n)\n"
    "from agent.pipeline.store import PipelineStore\n\n"
    '__all__ = ["PipelineStore", "PipelineScheduler", "StorageBackend", "SQLiteBackend", "PostgresBackend"]\n\n\n'
    "def __getattr__(name: str):\n"
    '    if name == "PipelineScheduler":\n'
    "        from agent.pipeline.scheduler import PipelineScheduler\n\n"
    "        return PipelineScheduler\n"
    '    raise AttributeError(f"module {__name__!r} has no attribute {name!r}")\n',
    encoding="utf-8",
)
print("✓ Patched agent.pipeline.__init__.py for lazy scheduler import")
print("\nSetup complete.")


## 3. Environment Check — GPU, PyTorch, PyG Versions


In [ ]:
import torch
import torch_geometric

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch      : {torch.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name     : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory // (1024**3)
    print(f"GPU VRAM     : {total_mem} GB")
else:
    print("GPU name     : none")
    print("GPU VRAM     : 0 GB")
print(f"PyG          : {torch_geometric.__version__}")
print(f"Device       : {DEVICE}")

if DEVICE == "cpu":
    print("Running in CPU fallback mode. This is supported, but it will be slower.")


## 4. Pre-flight Check — Verify Required Files Exist


In [ ]:
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
PIPELINE_DB = WORK_DIR / ".tirra_pipeline" / "pipeline.db"
CKPT_DIR = WORK_DIR / ".tirra_pipeline" / "checkpoints"

# Find the highest available epoch checkpoint
checkpoints = sorted(CKPT_DIR.glob("epoch_*.pt"))
assert checkpoints, (
    f"No checkpoints found in {CKPT_DIR}. "
    "This notebook is intended to resume from an existing epoch checkpoint."
)

latest_ckpt = checkpoints[-1]
resume_epoch = int(latest_ckpt.stem.split("_")[1])  # epoch_003 -> 3
print(f"Latest checkpoint : {latest_ckpt.name}  ({latest_ckpt.stat().st_size // 1_000_000} MB)")

# Verify pipeline DB
assert PIPELINE_DB.exists(), f"pipeline.db not found at {PIPELINE_DB}"
print(f"pipeline.db       : {PIPELINE_DB.stat().st_size // 1_000_000} MB")

# Verify key source files used by retrain_gnn.py and Trainer
for rel_path in [
    "agent/models/gnn/graph_builder.py",
    "agent/models/gnn/het_tgn.py",
    "agent/models/gnn/trainer.py",
    "agent/models/gnn/temporal.py",
    "agent/models/gnn/ewc.py",
    "agent/models/gnn/alignment.py",
    "scripts/retrain_gnn.py",
    "agent/pipeline/store.py",
]:
    p = WORK_DIR / rel_path
    assert p.exists(), f"Missing source file: {rel_path}"
    print(f"  ✓ {rel_path}")

print(f"\nAll checks passed. Will resume from epoch {resume_epoch}.")
RESUME_EPOCH = resume_epoch


## 5. Run GNN Training

Runs `scripts/retrain_gnn.py` as a subprocess so tqdm progress bars render correctly.
Training config: **10 total epochs, resume from latest checkpoint, weekly windows (2026 data), auto device selection**.

> **What changed since epochs 1–5:** `_contrastive_loss()` now L2-normalises embeddings before computing pairwise distances, so the margin=1.0 is meaningful. Contrastive signal was silently zero for all previous epochs — it will be active starting epoch 6.

> If a GPU is attached, the notebook uses CUDA.
> If not, it falls back to CPU automatically.

> Expected time:
> - T4 GPU: roughly 5–8 min per epoch

> - CPU fallback: slower, depending on the Kaggle host or local machine

In [ ]:
import subprocess
import sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
CKPT_DIR = WORK_DIR / ".tirra_pipeline" / "checkpoints"
DEVICE = globals().get("DEVICE", "cpu")

resume_epoch = globals().get("RESUME_EPOCH")
if resume_epoch is None:
    checkpoints = sorted(CKPT_DIR.glob("epoch_*.pt"))
    assert checkpoints, f"No checkpoints found in {CKPT_DIR}"
    resume_epoch = int(checkpoints[-1].stem.split("_")[1])
    print(f"Recovered RESUME_EPOCH={resume_epoch} from checkpoint directory.")

print(f"Using device: {DEVICE}")

# ── Training command ──────────────────────────────────────────────────────────
cmd = [
    sys.executable, "scripts/retrain_gnn.py",
    "--epochs",          "10",
    "--hidden-dim",      "128",
    "--num-layers",      "2",
    "--lr",              "1e-3",
    "--auto-tune",
    "--backup",
    "--since",           "2026-01-01",
    "--window-size",     "604800",     # weekly windows
    "--device",          DEVICE,
    "--skip-eval",
    "--checkpoint-dir",  str(CKPT_DIR),
    "--resume",          str(resume_epoch),
]

print("Running:", " ".join(cmd))
print("Working dir:", WORK_DIR)
print("-" * 70)

process = subprocess.Popen(
    cmd,
    cwd=str(WORK_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")

print("Training completed successfully.")


## 6. Verify Outputs and Prepare for Download


In [ ]:
import shutil
from pathlib import Path

WORK_DIR   = Path("/kaggle/working/tirramind_v1")
CKPT_DIR   = WORK_DIR / ".tirra_pipeline" / "checkpoints"
OUT_DIR    = Path("/kaggle/working")  # Kaggle auto-exposes this for download

print("=== Checkpoints saved ===")
for ckpt in sorted(CKPT_DIR.glob("epoch_*.pt")):
    size_mb = ckpt.stat().st_size / 1_000_000
    print(f"  {ckpt.name}  ({size_mb:.1f} MB)")

# Copy final model and all new epoch checkpoints to /kaggle/working/ for easy download
final_model = WORK_DIR / ".tirra_pipeline" / "gnn_model.pt"
if final_model.exists():
    shutil.copy2(final_model, OUT_DIR / "gnn_model.pt")
    print(f"\ngnn_model.pt copied to /kaggle/working/ ({final_model.stat().st_size / 1_000_000:.1f} MB)")
else:
    print("\ngnn_model.pt not found — training may not have completed all 5 epochs yet.")

# Copy epoch checkpoints > 5 to /kaggle/working/ for download
for ckpt in sorted(CKPT_DIR.glob("epoch_*.pt")):
    epoch_num = int(ckpt.stem.split("_")[1])
    if epoch_num > 5:
        dest = OUT_DIR / ckpt.name
        shutil.copy2(ckpt, dest)
        print(f"{ckpt.name} copied to /kaggle/working/")

print("\n=== Download from the 'Output' tab on the right panel ===")
print("Files to grab:")
print("  gnn_model.pt       → copy to .tirra_pipeline/gnn_model.pt")
print("  epoch_006.pt       → copy to .tirra_pipeline/checkpoints/")
print("  epoch_007.pt       → copy to .tirra_pipeline/checkpoints/")
print("  epoch_008.pt       → copy to .tirra_pipeline/checkpoints/")

print("  epoch_009.pt       → copy to .tirra_pipeline/checkpoints/")
print("  epoch_010.pt       → copy to .tirra_pipeline/checkpoints/")

## 7. Copy Trained Model Back to Laptop

After the Kaggle session finishes, download the output files and run these commands on your laptop:

```bash
cd /home/becmachlean/2024/projects/tirramind_v1

# Replace the final model
cp ~/Downloads/gnn_model.pt .tirra_pipeline/gnn_model.pt

# Add the new epoch checkpoints
cp ~/Downloads/epoch_006.pt .tirra_pipeline/checkpoints/
cp ~/Downloads/epoch_007.pt .tirra_pipeline/checkpoints/
cp ~/Downloads/epoch_008.pt .tirra_pipeline/checkpoints/
cp ~/Downloads/epoch_009.pt .tirra_pipeline/checkpoints/
cp ~/Downloads/epoch_010.pt .tirra_pipeline/checkpoints/

# Verify
ls -lh .tirra_pipeline/checkpoints/
ls -lh .tirra_pipeline/gnn_model.pt

# Run post-retrain diagnostic
python scripts/gnn_attention_diagnostic.py
```
